# 09 — Borough Comparison

Cross-borough analysis that runs **only when 2+ boroughs have been analyzed**.

**Outputs** (to `outputs/Comparison/`):
1. Comparison bar charts — cell counts, class distribution %, accuracy per borough
2. Combined map — all boroughs’ heatmaps on one map
3. Side-by-side per-borough diagrams — heatmaps and dashboards in a single sheet

In [ ]:
# ── Papermill parameters ────────────────────────────────────────
PLOTS_DIR = "outputs/Comparison"

In [ ]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
import json
import math
import os

os.makedirs(PLOTS_DIR, exist_ok=True)

with open("grid.json", encoding="utf-8") as f:
    config = json.load(f)

CELL_SIZE_M = config["grid_cell_size_m"]
INCLUDE_OTHER = config.get("include_other", False)
CLASS_ORDER = ["Commercial", "Residential", "Other"] if INCLUDE_OTHER else ["Commercial", "Residential"]
CLASS_COLS = {"Commercial": "#B2182B", "Residential": "#2166AC", "Other": "#1B7837"}

In [ ]:
# ── Discover all borough folders with prediction results ─
# Folder names are like "Manhattan_2026-05-15_14h30" or "Brooklyn_Queens_2026-05-15_14h30"
# We extract the borough prefix, group by it, and keep only the latest per borough.

import re

csv_root = Path("csv")
_TS_PATTERN = re.compile(r"^(.+)_(\d{4}-\d{2}-\d{2}_\d{2}h\d{2})$")

# Collect all folders with predictions, grouped by borough prefix
_by_borough = {}
for folder in sorted(csv_root.iterdir()):
    if folder.is_dir() and (folder / "07_predictions.csv").exists():
        m = _TS_PATTERN.match(folder.name)
        borough_key = m.group(1) if m else folder.name
        # Sorted iteration means last seen = latest timestamp
        _by_borough[borough_key] = folder

print("Latest run per borough:")
boroughs = {}
for borough_key, folder in sorted(_by_borough.items()):
    df = pd.read_csv(folder / "07_predictions.csv", dtype={"cell_id": str})
    if len(df) > 0:
        boroughs[borough_key] = {"df": df, "csv_folder": folder}
        print(f"  {borough_key}: {folder.name} ({len(df)} cells)")

print(f"\nTotal boroughs found: {len(boroughs)}")

if len(boroughs) < 2:
    print("\nSkipping comparison \u2014 need at least 2 analyzed boroughs.")
    raise SystemExit(0)

In [ ]:
# ── Build comparison dataframe ───────────────────────

_CLASS_MAP = {
    "Commercial": "Commercial",
    "Mixed-Use": "Residential",
    "Residential": "Residential",
}
if INCLUDE_OTHER:
    _CLASS_MAP.update({
        "Institutional": "Other", "Open Space": "Other",
        "Industrial": "Other", "Infrastructure": "Other",
    })

# Also resolve the latest outputs folder per borough (for loading PNGs)
_out_root = Path("outputs")
_out_by_borough = {}
for folder in sorted(_out_root.iterdir()):
    if folder.is_dir():
        m = _TS_PATTERN.match(folder.name)
        borough_key = m.group(1) if m else folder.name
        _out_by_borough[borough_key] = folder

summary_rows = []
for name, info in boroughs.items():
    df = info["df"]
    info["outputs_folder"] = _out_by_borough.get(name)
    df["actual_class"] = df["zone_type"].map(_CLASS_MAP)
    accuracy = (df["predicted_zone"] == df["actual_class"]).mean()
    row = {"borough": name, "total_cells": len(df), "accuracy": accuracy}
    for cls in CLASS_ORDER:
        n_pred = (df["predicted_zone"] == cls).sum()
        n_actual = (df["actual_class"] == cls).sum()
        row[f"pred_{cls}"] = n_pred
        row[f"pct_pred_{cls}"] = 100 * n_pred / len(df)
        row[f"actual_{cls}"] = n_actual
        row[f"pct_actual_{cls}"] = 100 * n_actual / len(df)
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

In [ ]:
# ── Plot 1: Comparison bar charts ────────────────────

n_boroughs = len(df_summary)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f"Borough Comparison \u2014 {n_boroughs} boroughs \u00b7 {CELL_SIZE_M}m grid",
             fontsize=14, fontweight="bold")

b_names = df_summary["borough"].tolist()
x = np.arange(n_boroughs)

# (0,0) Total cells per borough
ax = axes[0, 0]
ax.bar(x, df_summary["total_cells"], color="#555555", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(b_names, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Grid cells")
ax.set_title("Total Grid Cells")
for i, v in enumerate(df_summary["total_cells"]):
    ax.text(i, v + 20, str(v), ha="center", fontsize=9)

# (0,1) Accuracy per borough
ax = axes[0, 1]
colors_acc = ["#2ca02c" if a >= 0.8 else "#ff7f0e" if a >= 0.7 else "#d62728"
              for a in df_summary["accuracy"]]
ax.bar(x, df_summary["accuracy"] * 100, color=colors_acc, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(b_names, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Overall Accuracy")
ax.set_ylim(0, 105)
for i, v in enumerate(df_summary["accuracy"]):
    ax.text(i, v * 100 + 1, f"{v:.1%}", ha="center", fontsize=9)

# (1,0) Predicted class distribution (stacked %)
ax = axes[1, 0]
bottom = np.zeros(n_boroughs)
for cls in CLASS_ORDER:
    vals = df_summary[f"pct_pred_{cls}"].values
    ax.bar(x, vals, bottom=bottom, color=CLASS_COLS[cls], label=cls, alpha=0.85)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 5:
            ax.text(i, b + v / 2, f"{v:.0f}%", ha="center", va="center", fontsize=8, color="white", fontweight="bold")
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels(b_names, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Percentage")
ax.set_title("Predicted Class Distribution")
ax.legend(fontsize=9)
ax.set_ylim(0, 105)

# (1,1) Actual class distribution (stacked %)
ax = axes[1, 1]
bottom = np.zeros(n_boroughs)
for cls in CLASS_ORDER:
    vals = df_summary[f"pct_actual_{cls}"].values
    ax.bar(x, vals, bottom=bottom, color=CLASS_COLS[cls], label=cls, alpha=0.85)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 5:
            ax.text(i, b + v / 2, f"{v:.0f}%", ha="center", va="center", fontsize=8, color="white", fontweight="bold")
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels(b_names, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Percentage")
ax.set_title("Actual Class Distribution (PLUTO)")
ax.legend(fontsize=9)
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/01_borough_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {PLOTS_DIR}/01_borough_comparison.png")

In [ ]:
# ── Plot 2: Combined map — all boroughs on one figure ─

import contextily as ctx

# Merge all prediction DataFrames
all_dfs = []
for name, info in boroughs.items():
    df_copy = info["df"].copy()
    df_copy["borough"] = name
    all_dfs.append(df_copy)
df_all = pd.concat(all_dfs, ignore_index=True)

REF_LAT = df_all["cell_lat"].mean()
LAT_STEP = CELL_SIZE_M / 111_000
LON_STEP = CELL_SIZE_M / (111_000 * math.cos(math.radians(REF_LAT)))
HALF_LAT = LAT_STEP / 2
HALF_LON = LON_STEP / 2

# Determine figure aspect ratio from data bounds
lat_range = df_all["cell_lat"].max() - df_all["cell_lat"].min()
lon_range = df_all["cell_lon"].max() - df_all["cell_lon"].min()
aspect = lat_range / (lon_range * math.cos(math.radians(REF_LAT))) if lon_range > 0 else 1.5
fig_w = max(12, min(20, 14))
fig_h = max(8, min(22, fig_w * aspect))

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

if INCLUDE_OTHER:
    for _, row in df_all.iterrows():
        base_color = CLASS_COLS.get(row["predicted_zone"], "#999999")
        conf = row.get("confidence", 0.7)
        alpha = 0.4 + 0.5 * conf
        rect = mpatches.Rectangle(
            (row["cell_lon"] - HALF_LON, row["cell_lat"] - HALF_LAT),
            LON_STEP, LAT_STEP, linewidth=0.05, edgecolor="gray",
            facecolor=base_color, alpha=alpha)
        ax.add_patch(rect)
else:
    cmap = LinearSegmentedColormap.from_list("res_com", ["#2166AC", "#F7F7F7", "#B2182B"])
    for _, row in df_all.iterrows():
        prob_com = row.get("prob_commercial", 0.5)
        rect = mpatches.Rectangle(
            (row["cell_lon"] - HALF_LON, row["cell_lat"] - HALF_LAT),
            LON_STEP, LAT_STEP, linewidth=0.05, edgecolor="gray",
            facecolor=cmap(prob_com), alpha=0.75)
        ax.add_patch(rect)

pad = 0.005
ax.set_xlim(df_all["cell_lon"].min() - pad, df_all["cell_lon"].max() + pad)
ax.set_ylim(df_all["cell_lat"].min() - pad, df_all["cell_lat"].max() + pad)
ax.set_aspect("equal")

try:
    ctx.add_basemap(ax, crs="EPSG:4326",
                    source=ctx.providers.CartoDB.PositronNoLabels,
                    zoom=12, alpha=0.4)
except Exception as e:
    print(f"Basemap download failed: {e}")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

borough_list = ", ".join(boroughs.keys())
ax.set_title(
    f"Combined Heatmap \u2014 {borough_list}\n"
    f"{len(df_all)} total cells ({CELL_SIZE_M}m grid)",
    fontsize=13)

if INCLUDE_OTHER:
    legend_patches = [mpatches.Patch(color=CLASS_COLS[c], label=c) for c in CLASS_ORDER]
    ax.legend(handles=legend_patches, loc="lower right", fontsize=10,
              framealpha=0.9, edgecolor="gray")
else:
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.3, pad=0.02)
    cbar.set_label("P(Commercial)", fontsize=10)

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/02_combined_map.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {PLOTS_DIR}/02_combined_map.png")

In [ ]:
# ── Combined interactive folium map ───────────────────

try:
    import folium
    from folium import Rectangle

    m = folium.Map(
        location=[df_all["cell_lat"].mean(), df_all["cell_lon"].mean()],
        zoom_start=11,
        tiles="CartoDB positron",
    )

    for _, row in df_all.iterrows():
        bounds = [
            [row["cell_lat"] - HALF_LAT, row["cell_lon"] - HALF_LON],
            [row["cell_lat"] + HALF_LAT, row["cell_lon"] + HALF_LON],
        ]
        prob_com = row.get("prob_commercial", 0.5)

        if INCLUDE_OTHER:
            color = CLASS_COLS.get(row["predicted_zone"], "#999999")
            conf = row.get("confidence", 0.7)
            opacity = 0.5 + 0.4 * conf
            popup_text = (f"<b>{row['cell_id']}</b><br>"
                          f"Borough: {row['borough']}<br>"
                          f"Actual: {row['zone_type']}<br>"
                          f"Predicted: <b>{row['predicted_zone']}</b><br>"
                          f"Confidence: {conf:.0%}")
        else:
            color = "#B2182B" if prob_com >= 0.5 else "#2166AC"
            opacity = 0.4 + 0.5 * abs(prob_com - 0.5) * 2
            popup_text = (f"<b>{row['cell_id']}</b><br>"
                          f"Borough: {row['borough']}<br>"
                          f"Actual: {row['zone_type']}<br>"
                          f"Predicted: <b>{row['predicted_zone']}</b><br>"
                          f"P(Commercial): {prob_com:.0%}")

        Rectangle(
            bounds=bounds,
            color="gray", weight=0.2,
            fill=True, fill_color=color, fill_opacity=opacity,
            popup=folium.Popup(popup_text, max_width=200),
        ).add_to(m)

    html_path = f"{PLOTS_DIR}/02_combined_map_interactive.html"
    m.save(html_path)
    print(f"Saved: {html_path}")

except ImportError:
    print("folium not installed \u2014 skipping interactive map.")

In [ ]:
# ── Plot 3: Side-by-side heatmaps per borough ────────

b_names = list(boroughs.keys())

# Load per-borough heatmap PNGs from their latest outputs folder
heatmap_images = {}
for name, info in boroughs.items():
    out_folder = info.get("outputs_folder")
    if out_folder:
        img_path = out_folder / "08_heatmap_predictions.png"
        if img_path.exists():
            heatmap_images[name] = mpimg.imread(str(img_path))

if heatmap_images:
    n_imgs = len(heatmap_images)
    ncols = min(n_imgs, 3)
    nrows = math.ceil(n_imgs / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 12 * nrows))
    if n_imgs == 1:
        axes = np.array([axes])
    axes = np.atleast_2d(axes)

    fig.suptitle("Per-Borough Heatmaps", fontsize=16, fontweight="bold")

    for i, (name, img) in enumerate(heatmap_images.items()):
        r, c = divmod(i, ncols)
        ax = axes[r, c]
        ax.imshow(img)
        ax.set_title(name, fontsize=14)
        ax.axis("off")

    # Hide unused subplots
    for j in range(n_imgs, nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r, c].axis("off")

    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/03_heatmaps_side_by_side.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {PLOTS_DIR}/03_heatmaps_side_by_side.png")
else:
    print("No per-borough heatmap PNGs found \u2014 skipping side-by-side heatmaps.")

In [ ]:
# ── Plot 4: Side-by-side dashboards per borough ───────

dashboard_images = {}
for name, info in boroughs.items():
    out_folder = info.get("outputs_folder")
    if out_folder:
        img_path = out_folder / "09_summary_dashboard.png"
        if img_path.exists():
            dashboard_images[name] = mpimg.imread(str(img_path))

if dashboard_images:
    n_imgs = len(dashboard_images)
    ncols = min(n_imgs, 2)
    nrows = math.ceil(n_imgs / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14 * ncols, 10 * nrows))
    if n_imgs == 1:
        axes = np.array([axes])
    axes = np.atleast_2d(axes)

    fig.suptitle("Per-Borough Dashboards", fontsize=16, fontweight="bold")

    for i, (name, img) in enumerate(dashboard_images.items()):
        r, c = divmod(i, ncols)
        ax = axes[r, c]
        ax.imshow(img)
        ax.set_title(name, fontsize=14)
        ax.axis("off")

    for j in range(n_imgs, nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r, c].axis("off")

    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/04_dashboards_side_by_side.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {PLOTS_DIR}/04_dashboards_side_by_side.png")
else:
    print("No per-borough dashboard PNGs found \u2014 skipping side-by-side dashboards.")

In [ ]:
# ── Plot 5: Feature means comparison across boroughs ──

FEATURE_COLS = ["amenity_density", "shop_density_km2", "shop_type_entropy",
                "brand_ratio", "tourism_density", "landuse_entropy",
                "amenity_ratio_food_drink", "avg_floors", "avg_yearbuilt",
                "building_count", "total_bldg_area"]

# Compute mean features per borough (only those present in data)
_all_dfs = {name: info["df"] for name, info in boroughs.items()}
available_feats = [f for f in FEATURE_COLS if all(f in df.columns for df in _all_dfs.values())]

if available_feats:
    feat_means = {}
    for name, df in _all_dfs.items():
        feat_means[name] = df[available_feats].mean()
    df_feats = pd.DataFrame(feat_means).T

    # Normalize each feature to [0,1] for comparison
    df_norm = (df_feats - df_feats.min()) / (df_feats.max() - df_feats.min() + 1e-9)

    fig, ax = plt.subplots(figsize=(14, max(6, len(available_feats) * 0.6)))
    y_pos = np.arange(len(available_feats))
    bar_h = 0.8 / len(boroughs)
    cmap_boroughs = plt.cm.Set2(np.linspace(0, 1, len(boroughs)))

    for i, (name, row) in enumerate(df_norm.iterrows()):
        ax.barh(y_pos + i * bar_h, row[available_feats].values,
                bar_h, label=name, color=cmap_boroughs[i], alpha=0.85)

    ax.set_yticks(y_pos + bar_h * len(boroughs) / 2)
    ax.set_yticklabels(available_feats, fontsize=9)
    ax.set_xlabel("Normalized mean (0 = min borough, 1 = max borough)")
    ax.set_title("Feature Means Comparison Across Boroughs (normalized)", fontsize=13)
    ax.legend(fontsize=10)

    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/05_feature_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {PLOTS_DIR}/05_feature_comparison.png")
else:
    print("No common feature columns found \u2014 skipping feature comparison.")

In [ ]:
# ── Summary table ────────────────────────────────────

summary_path = f"{PLOTS_DIR}/comparison_summary.csv"
df_summary.to_csv(summary_path, index=False, encoding="utf-8")
print(f"Saved: {summary_path}")

print(f"\n{'='*55}")
print(f"  COMPARISON COMPLETE")
print(f"{'='*55}")
print(f"  Boroughs compared: {', '.join(boroughs.keys())}")
print(f"  Total cells: {sum(len(info['df']) for info in boroughs.values())}")
print(f"  Output folder: {PLOTS_DIR}/")
print(f"{'='*55}")